In [0]:
import logging

from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    desc,
    dense_rank,
    rank as sparkRank,
    row_number,
    to_timestamp,
    year
)


logger = logging.getLogger("silver-window-transformations")
logger.setLevel(logging.INFO)


class WindowTransformations:
    def __init__(
        self,
        dateColumnName: str = "order_date",
        amountColumnName: str = "total_amount"
    ):
        self.dateColumnName = dateColumnName
        self.amountColumnName = amountColumnName

    def renameRescuedDataColumn(self, df: DataFrame) -> DataFrame:
        if "_rescued_data" in df.columns:
            return df.withColumnRenamed("_rescued_data", "rescued_data")

        return df

    def dropRescuedDataColumn(self, df: DataFrame) -> DataFrame:
        if "rescued_data" in df.columns:
            return df.drop("rescued_data")

        return df

    def convertOrderDateToTimestamp(self, df: DataFrame) -> DataFrame:
        return df.withColumn(
            self.dateColumnName,
            to_timestamp(col(self.dateColumnName))
        )

    def addYearColumn(self, df: DataFrame) -> DataFrame:
        return df.withColumn(
            "year",
            year(col(self.dateColumnName))
        )

    def addDenseRankColumn(self, df: DataFrame) -> DataFrame:
        windowSpec = self.buildYearAmountWindow()

        return df.withColumn(
            "flag",
            dense_rank().over(windowSpec)
        )

    def addRankColumn(self, df: DataFrame) -> DataFrame:
        windowSpec = self.buildYearAmountWindow()

        return df.withColumn(
            "rank_flag",
            sparkRank().over(windowSpec)
        )

    def addRowNumberColumn(self, df: DataFrame) -> DataFrame:
        windowSpec = self.buildYearAmountWindow()

        return df.withColumn(
            "row_flag",
            row_number().over(windowSpec)
        )

    def transform(self, df: DataFrame) -> DataFrame:
        logger.info("Starting silver transformations")

        transformedDf = self.renameRescuedDataColumn(df)
        transformedDf = self.dropRescuedDataColumn(transformedDf)
        transformedDf = self.convertOrderDateToTimestamp(transformedDf)
        transformedDf = self.addYearColumn(transformedDf)
        transformedDf = self.addDenseRankColumn(transformedDf)
        transformedDf = self.addRankColumn(transformedDf)
        transformedDf = self.addRowNumberColumn(transformedDf)

        logger.info("Silver transformations completed")

        return transformedDf

    def buildYearAmountWindow(self):
        return (
            Window
            .partitionBy("year")
            .orderBy(desc(self.amountColumnName))
        )

In [0]:
bronzePath = (
    "abfss://bronze@datalakezakariae2026.dfs.core.windows.net/orders"
)

df = (
    spark.read
    .format("parquet")
    .load(bronzePath)
)

transformer = WindowTransformations(
    dateColumnName="order_date",
    amountColumnName="total_amount"
)

df1 = transformer.transform(df)

display(df1)

order_id,customer_id,product_id,order_date,quantity,total_amount,year,flag,rank_flag,row_flag
O00957,C01449,P0498,2023-10-05T00:00:00.000Z,5,9952.9,2023,1,1,1
O01765,C01515,P0498,2023-05-11T00:00:00.000Z,5,9952.9,2023,1,1,2
O03502,C00805,P0498,2023-09-24T00:00:00.000Z,5,9952.9,2023,1,1,3
O03660,C01001,P0498,2023-10-08T00:00:00.000Z,5,9952.9,2023,1,1,4
O06790,C01819,P0498,2023-02-27T00:00:00.000Z,5,9952.9,2023,1,1,5
O03989,C00631,P0440,2023-11-20T00:00:00.000Z,5,9916.25,2023,2,6,6
O07763,C00471,P0440,2023-04-12T00:00:00.000Z,5,9916.25,2023,2,6,7
O03272,C01879,P0165,2023-02-15T00:00:00.000Z,5,9858.85,2023,3,8,8
O03746,C01713,P0165,2023-10-26T00:00:00.000Z,5,9858.85,2023,3,8,9
O08261,C00988,P0165,2023-11-29T00:00:00.000Z,5,9858.85,2023,3,8,10


In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@datalakezakariae2026.dfs.core.windows.net/orders")